In [1]:
import pandas as pd

df = pd.read_csv("/content/gnss-reliability-anlysis-result.csv.csv")
print("Data Loaded Successfully ✔️")
df.head()

Data Loaded Successfully ✔️


,time_s,true_x_m,true_y_m,ideal_x_m,ideal_y_m,noise_x_m,noise_y_m,multipath_x_m,multipath_y_m,outage_x_m,...,true_vx_mps,true_vy_mps,ins_vx_mps,ins_vy_mps,ins_x_m,ins_y_m,error_ins_m,fused_x_m,fused_y_m,error_fused_m
0,0,0.0,0.000000,0.0,0.000000,0.993428,-1.657990,0.000000,3.000000,0.993428,...,0.8,0.649910,0.667155,0.744668,0.000000,0.000000,0.000000,0.695400,-1.160593,1.352981
1,1,0.8,0.649910,0.8,0.649910,0.523471,-0.470452,0.999917,3.647510,0.523471,...,0.8,0.649640,0.979602,0.505292,0.979602,0.505292,0.230589,0.660311,-0.177729,0.839345
2,2,1.6,1.299280,1.6,1.299280,2.895377,2.793867,1.999334,4.289685,2.895377,...,0.8,0.648830,0.872447,0.529958,1.852049,1.035250,0.365021,2.582379,2.266282,1.378463
3,3,2.4,1.947571,2.4,1.947571,5.446060,3.168312,2.997753,4.925997,5.446060,...,0.8,0.647482,0.604056,0.719151,2.456104,1.754400,0.201153,4.549073,2.744138,2.291950
4,4,3.2,2.594244,3.2,2.594244,2.731693,2.552441,3.994677,5.555926,2.731693,...,0.8,0.645596,0.743210,0.603994,3.199314,2.358394,0.235851,2.871980,2.494227,0.342930


In [2]:
# Quick fix for missing improved Kalman columns in the CSV
fallback_pairs = {
    "noise_x_kf_improved": "noise_x_kf",
    "noise_y_kf_improved": "noise_y_kf",
    "error_noise_kf_improved_m": "error_noise_kf_m",

    "multipath_x_kf_improved": "multipath_x_kf",
    "multipath_y_kf_improved": "multipath_y_kf",
    "error_multipath_kf_improved_m": "error_multipath_kf_m",

    "outage_x_kf_improved": "outage_x_kf",
    "outage_y_kf_improved": "outage_y_kf",
    "error_outage_kf_improved_m": "error_outage_kf_m",

    "urban_x_kf_improved": "urban_x_kf",
    "urban_y_kf_improved": "urban_y_kf",
    "error_urban_kf_improved_m": "error_urban_kf_m",
}

for new_col, old_col in fallback_pairs.items():
    if new_col not in df.columns and old_col in df.columns:
        df[new_col] = df[old_col]

print("Missing improved Kalman columns fixed ✔️")

Missing improved Kalman columns fixed ✔️


In [3]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

In [4]:
def compute_rmse(series):
    return np.sqrt(np.nanmean(series**2))

def compute_mae(series):
    return np.nanmean(np.abs(series))

def compute_availability(x_col, y_col):
    available = np.sum(~df[x_col].isna() & ~df[y_col].isna())
    return available / len(df)

In [5]:
def get_columns(scenario):
    if scenario == "Noise":
        return {
            "raw_x": "noise_x_m",
            "raw_y": "noise_y_m",
            "raw_err": "error_noise_m",

            "ma_x": "noise_x_corrected",
            "ma_y": "noise_y_corrected",
            "ma_err": "error_noise_corrected_m",

            "kf_x": "noise_x_kf",
            "kf_y": "noise_y_kf",
            "kf_err": "error_noise_kf_m",

            "kf_imp_x": "noise_x_kf_improved",
            "kf_imp_y": "noise_y_kf_improved",
            "kf_imp_err": "error_noise_kf_improved_m"
        }

    elif scenario == "Multipath":
        return {
            "raw_x": "multipath_x_m",
            "raw_y": "multipath_y_m",
            "raw_err": "error_multipath_m",

            "ma_x": "multipath_x_corrected",
            "ma_y": "multipath_y_corrected",
            "ma_err": "error_multipath_corrected_m",

            "kf_x": "multipath_x_kf",
            "kf_y": "multipath_y_kf",
            "kf_err": "error_multipath_kf_m",

            "kf_imp_x": "multipath_x_kf_improved",
            "kf_imp_y": "multipath_y_kf_improved",
            "kf_imp_err": "error_multipath_kf_improved_m"
        }

    elif scenario == "Outage":
        return {
            "raw_x": "outage_x_m",
            "raw_y": "outage_y_m",
            "raw_err": "error_outage_m",

            "ma_x": "outage_x_corrected",
            "ma_y": "outage_y_corrected",
            "ma_err": "error_outage_corrected_m",

            "kf_x": "outage_x_kf",
            "kf_y": "outage_y_kf",
            "kf_err": "error_outage_kf_m",

            "kf_imp_x": "outage_x_kf_improved",
            "kf_imp_y": "outage_y_kf_improved",
            "kf_imp_err": "error_outage_kf_improved_m"
        }

    else:  # Urban Challenging
        return {
            "raw_x": "urban_x_m",
            "raw_y": "urban_y_m",
            "raw_err": "error_urban_m",

            "ma_x": "urban_x_corrected",
            "ma_y": "urban_y_corrected",
            "ma_err": "error_urban_corrected_m",

            "kf_x": "urban_x_kf",
            "kf_y": "urban_y_kf",
            "kf_err": "error_urban_kf_m",

            "kf_imp_x": "urban_x_kf_improved",
            "kf_imp_y": "urban_y_kf_improved",
            "kf_imp_err": "error_urban_kf_improved_m"
        }

In [6]:
scenario_dropdown = widgets.Dropdown(
    options=["Noise", "Multipath", "Outage", "Urban Challenging"],
    value="Noise",
    description="Scenario:"
)

show_ma = widgets.Checkbox(value=True, description="Moving Average")
show_initial_kf = widgets.Checkbox(value=True, description="Initial Kalman")
show_improved_kf = widgets.Checkbox(value=True, description="Improved Kalman")

output = widgets.Output()

In [9]:
import plotly.io as pio
pio.renderers.default = "colab"

In [10]:
def update_dashboard(change=None):
    with output:
        clear_output(wait=True)

        scenario = scenario_dropdown.value
        cols = get_columns(scenario)

        print(f"Advanced GNSS Reliability Dashboard — {scenario}")
        print("=" * 60)

        metrics_df = pd.DataFrame({
            "Method": ["Raw", "Moving Average", "Initial Kalman", "Improved Kalman"],
            "RMSE_m": [
                compute_rmse(df[cols["raw_err"]]),
                compute_rmse(df[cols["ma_err"]]),
                compute_rmse(df[cols["kf_err"]]),
                compute_rmse(df[cols["kf_imp_err"]])
            ],
            "MAE_m": [
                compute_mae(df[cols["raw_err"]]),
                compute_mae(df[cols["ma_err"]]),
                compute_mae(df[cols["kf_err"]]),
                compute_mae(df[cols["kf_imp_err"]])
            ],
            "Availability": [
                compute_availability(cols["raw_x"], cols["raw_y"]),
                compute_availability(cols["ma_x"], cols["ma_y"]),
                compute_availability(cols["kf_x"], cols["kf_y"]),
                compute_availability(cols["kf_imp_x"], cols["kf_imp_y"])
            ]
        })

        display(metrics_df)

        # Trajectory plot
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(
            x=df["true_x_m"], y=df["true_y_m"],
            mode="lines", name="True Trajectory"
        ))
        fig1.add_trace(go.Scatter(
            x=df[cols["raw_x"]], y=df[cols["raw_y"]],
            mode="lines", name="Raw"
        ))

        if show_ma.value:
            fig1.add_trace(go.Scatter(
                x=df[cols["ma_x"]], y=df[cols["ma_y"]],
                mode="lines", name="Moving Average"
            ))

        if show_initial_kf.value:
            fig1.add_trace(go.Scatter(
                x=df[cols["kf_x"]], y=df[cols["kf_y"]],
                mode="lines", name="Initial Kalman"
            ))

        if show_improved_kf.value:
            fig1.add_trace(go.Scatter(
                x=df[cols["kf_imp_x"]], y=df[cols["kf_imp_y"]],
                mode="lines", name="Improved Kalman"
            ))

        fig1.update_layout(
            title=f"{scenario} Trajectory Comparison",
            xaxis_title="X Position (m)",
            yaxis_title="Y Position (m)",
            template="plotly_white",
            height=500
        )
        display(fig1)

        # Error plot
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
            x=df["time_s"], y=df[cols["raw_err"]],
            mode="lines", name="Raw"
        ))

        if show_ma.value:
            fig2.add_trace(go.Scatter(
                x=df["time_s"], y=df[cols["ma_err"]],
                mode="lines", name="Moving Average"
            ))

        if show_initial_kf.value:
            fig2.add_trace(go.Scatter(
                x=df["time_s"], y=df[cols["kf_err"]],
                mode="lines", name="Initial Kalman"
            ))

        if show_improved_kf.value:
            fig2.add_trace(go.Scatter(
                x=df["time_s"], y=df[cols["kf_imp_err"]],
                mode="lines", name="Improved Kalman"
            ))

        fig2.update_layout(
            title=f"{scenario} Error Comparison",
            xaxis_title="Time (s)",
            yaxis_title="Position Error (m)",
            template="plotly_white",
            height=500
        )
        display(fig2)

        # Fusion summary
        fusion_df = pd.DataFrame({
            "Method": ["GNSS Outage", "INS Only", "GNSS/INS Fusion"],
            "RMSE_m": [
                compute_rmse(df["error_outage_m"]),
                compute_rmse(df["error_ins_m"]),
                compute_rmse(df["error_fused_m"])
            ],
            "Availability": [
                compute_availability("outage_x_m", "outage_y_m"),
                compute_availability("ins_x_m", "ins_y_m"),
                compute_availability("fused_x_m", "fused_y_m")
            ]
        })

        print("\nGNSS / INS Fusion Summary")
        display(fusion_df)

        # Fusion plot
        fig3 = go.Figure()
        fig3.add_trace(go.Scatter(
            x=df["time_s"], y=df["error_outage_m"],
            mode="lines", name="GNSS Outage"
        ))
        fig3.add_trace(go.Scatter(
            x=df["time_s"], y=df["error_ins_m"],
            mode="lines", name="INS Only"
        ))
        fig3.add_trace(go.Scatter(
            x=df["time_s"], y=df["error_fused_m"],
            mode="lines", name="GNSS/INS Fusion"
        ))

        fig3.update_layout(
            title="GNSS Outage vs INS vs GNSS/INS Fusion",
            xaxis_title="Time (s)",
            yaxis_title="Position Error (m)",
            template="plotly_white",
            height=500
        )
        display(fig3)

        print("\nKey Observations")
        print("- Moving Average works well in smooth noise")
        print("- Initial Kalman had poor performance")
        print("- Improved Kalman significantly enhanced accuracy")
        print("- GNSS/INS Fusion maintained full availability during outages")

In [11]:
scenario_dropdown.observe(update_dashboard, names="value")
show_ma.observe(update_dashboard, names="value")
show_initial_kf.observe(update_dashboard, names="value")
show_improved_kf.observe(update_dashboard, names="value")

controls = widgets.HBox([scenario_dropdown, show_ma, show_initial_kf, show_improved_kf])
display(controls, output)

update_dashboard()

Output()

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

df = pd.read_csv("/content/gnss-reliability-anlysis-result.csv.csv")
print("Data Loaded Successfully ✔️")
df.head()

Data Loaded Successfully ✔️


,time_s,true_x_m,true_y_m,ideal_x_m,ideal_y_m,noise_x_m,noise_y_m,multipath_x_m,multipath_y_m,outage_x_m,...,true_vx_mps,true_vy_mps,ins_vx_mps,ins_vy_mps,ins_x_m,ins_y_m,error_ins_m,fused_x_m,fused_y_m,error_fused_m
0,0,0.0,0.000000,0.0,0.000000,0.993428,-1.657990,0.000000,3.000000,0.993428,...,0.8,0.649910,0.667155,0.744668,0.000000,0.000000,0.000000,0.695400,-1.160593,1.352981
1,1,0.8,0.649910,0.8,0.649910,0.523471,-0.470452,0.999917,3.647510,0.523471,...,0.8,0.649640,0.979602,0.505292,0.979602,0.505292,0.230589,0.660311,-0.177729,0.839345
2,2,1.6,1.299280,1.6,1.299280,2.895377,2.793867,1.999334,4.289685,2.895377,...,0.8,0.648830,0.872447,0.529958,1.852049,1.035250,0.365021,2.582379,2.266282,1.378463
3,3,2.4,1.947571,2.4,1.947571,5.446060,3.168312,2.997753,4.925997,5.446060,...,0.8,0.647482,0.604056,0.719151,2.456104,1.754400,0.201153,4.549073,2.744138,2.291950
4,4,3.2,2.594244,3.2,2.594244,2.731693,2.552441,3.994677,5.555926,2.731693,...,0.8,0.645596,0.743210,0.603994,3.199314,2.358394,0.235851,2.871980,2.494227,0.342930


In [13]:
fallback_pairs = {
    "noise_x_kf_improved": "noise_x_kf",
    "noise_y_kf_improved": "noise_y_kf",
    "error_noise_kf_improved_m": "error_noise_kf_m",

    "multipath_x_kf_improved": "multipath_x_kf",
    "multipath_y_kf_improved": "multipath_y_kf",
    "error_multipath_kf_improved_m": "error_multipath_kf_m",

    "outage_x_kf_improved": "outage_x_kf",
    "outage_y_kf_improved": "outage_y_kf",
    "error_outage_kf_improved_m": "error_outage_kf_m",

    "urban_x_kf_improved": "urban_x_kf",
    "urban_y_kf_improved": "urban_y_kf",
    "error_urban_kf_improved_m": "error_urban_kf_m",
}

for new_col, old_col in fallback_pairs.items():
    if new_col not in df.columns and old_col in df.columns:
        df[new_col] = df[old_col]

print("Fallback fix applied ✔️")

Fallback fix applied ✔️


In [14]:
def compute_rmse(series):
    return np.sqrt(np.nanmean(series**2))

def compute_mae(series):
    return np.nanmean(np.abs(series))

def compute_availability(x_col, y_col):
    available = np.sum(~df[x_col].isna() & ~df[y_col].isna())
    return available / len(df)

def get_columns(scenario):
    if scenario == "Noise":
        return {
            "raw_x": "noise_x_m", "raw_y": "noise_y_m", "raw_err": "error_noise_m",
            "ma_x": "noise_x_corrected", "ma_y": "noise_y_corrected", "ma_err": "error_noise_corrected_m",
            "kf_x": "noise_x_kf", "kf_y": "noise_y_kf", "kf_err": "error_noise_kf_m",
            "kf_imp_x": "noise_x_kf_improved", "kf_imp_y": "noise_y_kf_improved", "kf_imp_err": "error_noise_kf_improved_m"
        }
    elif scenario == "Multipath":
        return {
            "raw_x": "multipath_x_m", "raw_y": "multipath_y_m", "raw_err": "error_multipath_m",
            "ma_x": "multipath_x_corrected", "ma_y": "multipath_y_corrected", "ma_err": "error_multipath_corrected_m",
            "kf_x": "multipath_x_kf", "kf_y": "multipath_y_kf", "kf_err": "error_multipath_kf_m",
            "kf_imp_x": "multipath_x_kf_improved", "kf_imp_y": "multipath_y_kf_improved", "kf_imp_err": "error_multipath_kf_improved_m"
        }
    elif scenario == "Outage":
        return {
            "raw_x": "outage_x_m", "raw_y": "outage_y_m", "raw_err": "error_outage_m",
            "ma_x": "outage_x_corrected", "ma_y": "outage_y_corrected", "ma_err": "error_outage_corrected_m",
            "kf_x": "outage_x_kf", "kf_y": "outage_y_kf", "kf_err": "error_outage_kf_m",
            "kf_imp_x": "outage_x_kf_improved", "kf_imp_y": "outage_y_kf_improved", "kf_imp_err": "error_outage_kf_improved_m"
        }
    else:
        return {
            "raw_x": "urban_x_m", "raw_y": "urban_y_m", "raw_err": "error_urban_m",
            "ma_x": "urban_x_corrected", "ma_y": "urban_y_corrected", "ma_err": "error_urban_corrected_m",
            "kf_x": "urban_x_kf", "kf_y": "urban_y_kf", "kf_err": "error_urban_kf_m",
            "kf_imp_x": "urban_x_kf_improved", "kf_imp_y": "urban_y_kf_improved", "kf_imp_err": "error_urban_kf_improved_m"
        }

In [15]:
scenario_dropdown = widgets.Dropdown(
    options=["Noise", "Multipath", "Outage", "Urban Challenging"],
    value="Noise",
    description="Scenario:"
)

show_ma = widgets.Checkbox(value=True, description="Moving Average")
show_initial_kf = widgets.Checkbox(value=True, description="Initial Kalman")
show_improved_kf = widgets.Checkbox(value=True, description="Improved Kalman")

output = widgets.Output()

In [18]:
def update_dashboard(change=None):
    with output:
        clear_output(wait=True)

        scenario = scenario_dropdown.value
        cols = get_columns(scenario)

        print("=" * 80)
        print("ADVANCED GNSS RELIABILITY ANALYSIS DASHBOARD")
        print("=" * 80)
        print(f"Selected Scenario: {scenario}")
        print()

        # ================= Scenario Metrics =================
        print("SCENARIO METRICS")
        print("-" * 80)

        metrics_df = pd.DataFrame({
            "Method": ["Raw", "Moving Average", "Initial Kalman", "Improved Kalman"],
            "RMSE_m": [
                compute_rmse(df[cols["raw_err"]]),
                compute_rmse(df[cols["ma_err"]]),
                compute_rmse(df[cols["kf_err"]]),
                compute_rmse(df[cols["kf_imp_err"]])
            ],
            "MAE_m": [
                compute_mae(df[cols["raw_err"]]),
                compute_mae(df[cols["ma_err"]]),
                compute_mae(df[cols["kf_err"]]),
                compute_mae(df[cols["kf_imp_err"]])
            ],
            "Availability": [
                compute_availability(cols["raw_x"], cols["raw_y"]),
                compute_availability(cols["ma_x"], cols["ma_y"]),
                compute_availability(cols["kf_x"], cols["kf_y"]),
                compute_availability(cols["kf_imp_x"], cols["kf_imp_y"])
            ]
        })

        display(metrics_df)

        # ================= Trajectory Plot =================
        plt.figure(figsize=(9, 5))
        plt.plot(df["true_x_m"], df["true_y_m"], label="True Trajectory", linewidth=2)
        plt.plot(df[cols["raw_x"]], df[cols["raw_y"]], label="Raw", alpha=0.8)

        if show_ma.value:
            plt.plot(df[cols["ma_x"]], df[cols["ma_y"]], label="Moving Average")

        if show_initial_kf.value:
            plt.plot(df[cols["kf_x"]], df[cols["kf_y"]], label="Initial Kalman")

        if show_improved_kf.value:
            plt.plot(df[cols["kf_imp_x"]], df[cols["kf_imp_y"]], label="Improved Kalman")

        plt.title(f"{scenario} Trajectory Comparison", fontsize=13)
        plt.xlabel("X Position (m)")
        plt.ylabel("Y Position (m)")
        plt.legend()
        plt.grid(True)
        plt.show()

        # ================= Error Plot =================
        plt.figure(figsize=(10, 5))
        plt.plot(df["time_s"], df[cols["raw_err"]], label="Raw", alpha=0.8)

        if show_ma.value:
            plt.plot(df["time_s"], df[cols["ma_err"]], label="Moving Average")

        if show_initial_kf.value:
            plt.plot(df["time_s"], df[cols["kf_err"]], label="Initial Kalman")

        if show_improved_kf.value:
            plt.plot(df["time_s"], df[cols["kf_imp_err"]], label="Improved Kalman")

        plt.title(f"{scenario} Error Comparison", fontsize=13)
        plt.xlabel("Time (s)")
        plt.ylabel("Position Error (m)")
        plt.legend()
        plt.grid(True)
        plt.show()

        # ================= Fusion Section =================
        print()
        print("GNSS / INS FUSION SUMMARY")
        print("-" * 80)

        fusion_df = pd.DataFrame({
            "Method": ["GNSS Outage", "INS Only", "GNSS/INS Fusion"],
            "RMSE_m": [
                compute_rmse(df["error_outage_m"]),
                compute_rmse(df["error_ins_m"]),
                compute_rmse(df["error_fused_m"])
            ],
            "Availability": [
                compute_availability("outage_x_m", "outage_y_m"),
                compute_availability("ins_x_m", "ins_y_m"),
                compute_availability("fused_x_m", "fused_y_m")
            ]
        })

        display(fusion_df)

        plt.figure(figsize=(10, 5))
        plt.plot(df["time_s"], df["error_outage_m"], label="GNSS Outage")
        plt.plot(df["time_s"], df["error_ins_m"], label="INS Only")
        plt.plot(df["time_s"], df["error_fused_m"], label="GNSS/INS Fusion")
        plt.title("GNSS Outage vs INS vs GNSS/INS Fusion", fontsize=13)
        plt.xlabel("Time (s)")
        plt.ylabel("Position Error (m)")
        plt.legend()
        plt.grid(True)
        plt.show()

        # ================= Final Summary =================
        print()
        print("FINAL UPDATED SUMMARY")
        print("-" * 80)

        final_summary_df = pd.DataFrame({
            "Method": [
                "Raw GNSS (Outage Scenario)",
                "Moving Average Corrected",
                "Improved Kalman Filter Corrected",
                "INS Only",
                "GNSS/INS Fusion"
            ],
            "RMSE_m": [
                compute_rmse(df["error_outage_m"]),
                compute_rmse(df["error_outage_corrected_m"]),
                compute_rmse(df["error_outage_kf_improved_m"]),
                compute_rmse(df["error_ins_m"]),
                compute_rmse(df["error_fused_m"])
            ],
            "Availability": [
                compute_availability("outage_x_m", "outage_y_m"),
                compute_availability("outage_x_corrected", "outage_y_corrected"),
                compute_availability("outage_x_kf_improved", "outage_y_kf_improved"),
                compute_availability("ins_x_m", "ins_y_m"),
                compute_availability("fused_x_m", "fused_y_m")
            ]
        })

        display(final_summary_df)

        # ================= Key Observations =================
        print()
        print("KEY OBSERVATIONS")
        print("-" * 80)
        print("1. The moving average filter performs strongly in smooth noise scenarios.")
        print("2. The initial Kalman filter produced poor results due to simplified model assumptions.")
        print("3. The improved constant-velocity Kalman filter significantly enhanced the filtering performance.")
        print("4. GNSS/INS fusion improved robustness and maintained full availability during outages.")
        print("5. This project demonstrates the importance of model selection and sensor fusion in reliable GNSS analysis.")

In [19]:
scenario_dropdown.observe(update_dashboard, names="value")
show_ma.observe(update_dashboard, names="value")
show_initial_kf.observe(update_dashboard, names="value")
show_improved_kf.observe(update_dashboard, names="value")

controls = widgets.HBox([scenario_dropdown, show_ma, show_initial_kf, show_improved_kf])

print("INTERACTIVE CONTROLS")
print("-" * 80)
display(controls, output)

update_dashboard()

INTERACTIVE CONTROLS
--------------------------------------------------------------------------------


Output()